# Google Play Store Uygulama Başarı Tahmini
**Öğrenci:** Burak Eren  
**Yöntem:** Random Forest Classifier & Logistic Regression  
**Veri Seti:** Google Play Store Apps (Kaggle)

---
Bu notebook, bir mobil uygulamanın kullanıcılar tarafından yüksek puan (4.0+) alıp almayacağını tahmin eden bir sınıflandırma modeli içermektedir.

## 1. GÜN — Kütüphanelerin Yüklenmesi ve Veri Okuma

In [ ]:
import pandas as pd          # Tabloları (CSV) okumak ve üzerinde işlem yapmak için
import numpy as np           # Matematik işlemleri için (medyan, nan gibi şeyler burada)
import matplotlib.pyplot as plt  # Grafik çizmek için temel kütüphane
import seaborn as sns        # matplotlib'in üstüne kurulu, daha şık grafikler çizmek için
import warnings              # Gereksiz uyarı mesajlarını susturmak için
warnings.filterwarnings('ignore')  # Çalışırken çıkan uyarıları gösterme, ekranı kirletmesin

df = pd.read_csv('googleplaystore.csv')  # CSV dosyasını oku ve df adlı tabloya yükle (df = dataframe kısaltması)
print('Veri seti boyutu:', df.shape)     # Kaç satır, kaç sütun olduğunu göster
print('\nSütunlar ve veri tipleri:')
print(df.dtypes)  # Her sütunun tipi ne: sayı mı (int/float) metin mi (object)?

In [ ]:
df.head()  # Tablonun ilk 5 satırını göster — veriyi tanımak için ilk bakış

In [ ]:
print('Eksik değer sayıları:')
print(df.isnull().sum())  # Her sütunda kaç tane boş (NaN) değer var, tek tek say

## Adım 1.3 — Veri Temizleme

Verideki **Installs**, **Size** ve **Price** sütunları metin (string) formatındadır. Yapay zeka modeline verebilmek için bunları sayısal değerlere dönüştürmemiz gerekiyor.

In [ ]:
# --- Installs (İndirilme Sayısı) Temizliği ---
# Verisetinde indirilme sayıları '10,000+' gibi yazılmış — hem virgül hem artı var
# Python bu haliyle bunu sayı olarak göremez, önce o karakterlerden kurtulmak lazım

df['Installs'] = df['Installs'].str.replace('+', '', regex=False)  # '+' işaretini sil
df['Installs'] = df['Installs'].str.replace(',', '', regex=False)  # Binlik ayırıcı virgülü sil
df = df[df['Installs'] != 'Free']      # 'Free' yazan hatalı satırı at (bu bir sayı değil)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')  # Metni sayıya çevir
# errors='coerce' → çevrilemeyen değerleri silmek yerine NaN (boş) yap

print('Installs örnek değerler:', df['Installs'].head().tolist())

In [ ]:
# --- Price (Fiyat) Temizliği ---
# Fiyatlar '$2.99' şeklinde yazılmış, başındaki dolar işaretini kaldırmak yeterli

df['Price'] = df['Price'].str.replace('$', '', regex=False)  # '$' işaretini sil
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')    # Metni sayıya çevir

print('Price örnek değerler:', df['Price'].head().tolist())

In [ ]:
# --- Size (Boyut) Temizliği ---
# Boyutlar '19M' veya '512k' şeklinde — M=Megabyte, k=kilobyte
# İkisi farklı birim, hepsini MB'a çevirerek eşitlemeliyiz
# (1 MB = 1024 KB, bu yüzden k olanları 1024'e böleceğiz)

def clean_size(size_val):  # Her satıra uygulanacak özel temizlik fonksiyonu
    if 'M' in str(size_val):                            # Megabyte ise
        return float(str(size_val).replace('M', ''))    # sadece M harfini sil, sayıya çevir
    elif 'k' in str(size_val):                          # Kilobyte ise
        return float(str(size_val).replace('k', '')) / 1024  # k'yı sil, 1024'e böl → MB'a çevir
    return np.nan  # Ne M ne k varsa (örn. 'Varies with device') boş bırak

df['Size'] = df['Size'].apply(clean_size)
# apply() → 'bu fonksiyonu tablodaki her satır için tek tek çalıştır' demek

df['Size'] = df['Size'].fillna(df['Size'].median())
# fillna() → boş (NaN) değerleri doldur
# .median() → ortanca değeri kullandık çünkü ortalama (mean) aşırı büyük/küçük değerlerden etkilenir
# ortanca ise tam ortadaki değer, daha sağlam bir merkez noktası

print('Size medyan:', df['Size'].median(), 'MB')

In [ ]:
# Reviews sütununu sayısala çevir
# Yorum sayısı da metin olarak gelmiş olabilir, güvenli olmak için dönüştürüyoruz
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce').fillna(0)
# Boş olanları 0 yaptık çünkü yorum sayısı yoksa 0'dır, silinmesine gerek yok

print('Reviews istatistikleri:')
print(df['Reviews'].describe())  # Min, max, ortalama gibi genel istatistikleri göster

## Adım 1.4 — Hedef Değişkenin Oluşturulması

Projemiz bir **sınıflandırma** problemidir:
- **1 (Başarılı):** Rating ≥ 4.0  
- **0 (Başarısız):** Rating < 4.0

In [ ]:
df = df.dropna(subset=['Rating'])
# dropna() → belirtilen sütunda boş değer olan satırları tamamen sil
# Rating'i eksik olan satırı tutmanın anlamı yok çünkü tahmin etmeye çalıştığımız şey bu

df['Yüksek_Puan'] = (df['Rating'] >= 4.0).astype(int)
# (df['Rating'] >= 4.0) → her satır için True/False üretir
# .astype(int) → True'yu 1'e, False'u 0'a çevirir
# Sonuç: modele 'doğru cevap' olarak vereceğimiz sütun

print('Hedef değişken dağılımı:')
print(df['Yüksek_Puan'].value_counts())  # Kaç tane 1, kaç tane 0 var?
print(f'\nTemizlenmiş veri boyutu: {df.shape}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))  # 6x4 inç boyutunda boş bir grafik alanı oluştur
df['Yüksek_Puan'].value_counts().plot(kind='bar', ax=ax,
    color=['tomato', 'steelblue'], edgecolor='white')  # Sütun grafik çiz, renkleri belirle
ax.set_xticklabels(['Başarısız (0)', 'Başarılı (1)'], rotation=0)  # X eksenindeki etiketleri yaz
ax.set_title('Hedef Değişken Dağılımı')   # Grafik başlığı
ax.set_ylabel('Uygulama Sayısı')          # Y ekseninin etiketi
plt.tight_layout()  # Elemanların birbirine girmemesi için otomatik hizala
plt.show()          # Grafiği ekranda göster

---
## 2. GÜN — Model Eğitimi, Karşılaştırma ve Görselleştirme

## Adım 2.1 — Özellik Seçimi ve Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split  # Veriyi ikiye bölmek için
from sklearn.preprocessing import StandardScaler      # Sayıları aynı ölçeğe çekmek için

features = ['Reviews', 'Size', 'Installs', 'Price']  # Modele girdi olarak verilecek sütunlar
X = df[features].fillna(0)   # X = girdiler (modelin 'bakacağı' veriler)
y = df['Yüksek_Puan']        # y = doğru cevaplar (model bunları tahmin etmeye çalışacak)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,     # Verinin %20'sini test için kenara ayır, %80 eğitimde kullanılacak
    random_state=42    # Bölme işlemi her çalıştırmada aynı şekilde yapılsın (42 sihirli bir sayı değil, herhangi bir sabit olabilir)
)

print(f'Eğitim seti boyutu : {X_train.shape}')  # Modelin öğreneceği veri
print(f'Test seti boyutu   : {X_test.shape}')   # Modelin hiç görmediği, test için sakladığımız veri

## Adım 2.2 — Modellerin Kurulması ve Eğitilmesi

In [ ]:
from sklearn.linear_model import LogisticRegression      # 1. model
from sklearn.ensemble import RandomForestClassifier      # 2. model
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
# classification_report → precision, recall, f1 gibi tüm metrikleri tek seferde verir
# confusion_matrix → modelin hangi tahminlerde yanıldığını matris olarak gösterir
# accuracy_score → kaç tahminin doğru olduğunun yüzdesi
# f1_score → hem yanlış pozitifleri hem yanlış negatifleri dengeleyen tek bir skor

scaler = StandardScaler()  # Scaler oluştur (henüz bir şey yapmıyor, sadece hazırlandı)
X_train_scaled = scaler.fit_transform(X_train)
# fit_transform → eğitim verisine bak, ortalama ve sapmaları öğren, sonra ölçekle
# Neden ölçekliyoruz? Installs milyonlara ulaşıyor, Price ise genelde 0-10 arası
# Bu fark Logistic Regression'ı yanıltır, büyük sayıya fazla ağırlık verir
X_test_scaled = scaler.transform(X_test)
# Test verisine sadece transform (ölçekle) uygula, fit değil
# Çünkü scaler eğitim verisinden öğrendi, test verisine aynı kuralı uyguluyoruz

# --- Model 1: Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000, random_state=42)
# max_iter=1000 → modelin cevaba ulaşmak için deneyeceği maksimum adım sayısı
# varsayılan 100'dür ama bu veri için yetmeyebilir, 1000 yaptık
lr_model.fit(X_train_scaled, y_train)  # fit() → modeli eğit: girdi-çıktı eşleşmelerini öğren
lr_preds = lr_model.predict(X_test_scaled)  # predict() → test verisine bak, tahmin üret
print('Logistic Regression eğitimi tamamlandı.')

# --- Model 2: Random Forest ---
rf_model = RandomForestClassifier(
    n_estimators=100,  # 100 adet karar ağacı oluştur, her biri ayrı ayrı tahmin yapar
    random_state=42    # Sonuçların tekrarlanabilir olması için
)
# Neden Random Forest? Hem büyük (Installs) hem küçük (Price) sayılar bir arada
# Ağaç yapısı büyüklük farkından etkilenmiyor, Scaler'a gerek yok
rf_model.fit(X_train, y_train)   # Ham (ölçeklenmemiş) veriyle eğit
rf_preds = rf_model.predict(X_test)  # Test tahminlerini üret
print('Random Forest eğitimi tamamlandı.')

## Adım 2.3 — Sonuçların Değerlendirilmesi

In [ ]:
print('--- LOJİSTİK REGRESYON RAPORU ---')
print(classification_report(y_test, lr_preds, target_names=['Başarısız (0)', 'Başarılı (1)']))
# classification_report(gerçek_cevaplar, model_tahminleri)
# Çıktıdaki satırlar ne anlama gelir:
# precision → 'başarılı' dediğinde kaç tanesini gerçekten doğru söyledi?
# recall → gerçekte başarılı olanların kaçını bulabildi?
# f1-score → precision ve recall'ın dengeli ortalaması
# support → test setinde o sınıftan kaç tane var

print('\n--- RANDOM FOREST RAPORU ---')
print(classification_report(y_test, rf_preds, target_names=['Başarısız (0)', 'Başarılı (1)']))

In [ ]:
# İki modeli yan yana karşılaştır
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [
        round(accuracy_score(y_test, lr_preds), 4),  # Toplam doğru tahmin / toplam tahmin
        round(accuracy_score(y_test, rf_preds), 4)
    ],
    'F1-Score (Weighted)': [
        round(f1_score(y_test, lr_preds, average='weighted'), 4),
        # average='weighted' → her sınıfın skorunu, o sınıftaki örnek sayısına göre ağırlıklı ortala
        round(f1_score(y_test, rf_preds, average='weighted'), 4)
    ]
})
print(results.to_string(index=False))  # Satır numarası olmadan tabloyu yazdır

## Adım 2.4 — Confusion Matrix (Hata Matrisi)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# 1 satır 2 sütun grafik düzeni — her iki model yan yana görünsün
# figsize=(12,5) → toplam grafik alanının genişliği ve yüksekliği (inç)

for ax, preds, title in zip(axes, [lr_preds, rf_preds], ['Logistic Regression', 'Random Forest']):
    # zip() → iki listeyi eşleştirerek beraber döndürür (LR grafiki sol, RF grafiki sağ)
    cm = confusion_matrix(y_test, preds)
    # confusion_matrix → 2x2 matris döner:
    # [sol üst] gerçek 0, tahmin 0 (doğru) | [sağ üst] gerçek 0, tahmin 1 (yanlış)
    # [sol alt] gerçek 1, tahmin 0 (yanlış) | [sağ alt] gerçek 1, tahmin 1 (doğru)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Başarısız', 'Başarılı'],
        yticklabels=['Başarısız', 'Başarılı'])
    # annot=True → her hücrenin içine sayıyı yaz
    # fmt='d'   → sayıları tam sayı olarak göster (ondalıklı değil)
    # cmap='Blues' → renk skalası mavi tonlarında olsun
    ax.set_title(f'{title} — Hata Matrisi', fontsize=13, fontweight='bold')
    ax.set_ylabel('Gerçek Değerler')    # Y ekseni: gerçek etiketler
    ax.set_xlabel('Tahmin Edilenler')   # X ekseni: modelin tahminleri

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
# savefig() → grafiği dosyaya kaydet (ekranda göstermek yerine veya ek olarak)
# dpi=150 → çözünürlük (pixel yoğunluğu), yüksek = daha net görsel
# bbox_inches='tight' → grafiğin etrafındaki boşlukları kırp
plt.show()
print('confusion_matrix.png kaydedildi.')

In [ ]:
# Accuracy ve F1-Score karşılaştırma grafiği
fig, ax = plt.subplots(figsize=(8, 5))
models_names = ['Logistic Regression', 'Random Forest']
acc_vals = [accuracy_score(y_test, lr_preds), accuracy_score(y_test, rf_preds)]
f1_vals  = [f1_score(y_test, lr_preds, average='weighted'),
            f1_score(y_test, rf_preds, average='weighted')]

x = np.arange(len(models_names))  # [0, 1] — her model için bir konum
width = 0.35  # Her çubuğun genişliği (yan yana iki çubuk olacak)

bars1 = ax.bar(x - width/2, acc_vals, width, label='Accuracy',  color='steelblue')
bars2 = ax.bar(x + width/2, f1_vals,  width, label='F1-Score',  color='darkorange')
# x - width/2 → sol çubuğu biraz sola kaydır, x + width/2 → sağ çubuğu sağa kaydır

ax.set_ylim(0, 1.1)  # Y ekseni 0'dan 1.1'e kadar (bar üstüne yazı için biraz boşluk)
ax.set_ylabel('Skor')
ax.set_title('Model Performans Karşılaştırması')
ax.set_xticks(x)                      # X ekseninde tick konumlarını belirle
ax.set_xticklabels(models_names)      # O konumlara model isimlerini yaz
ax.legend()                           # Renk açıklamalarını (Accuracy / F1-Score) göster

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2,  # Yazıyı çubuğun ortasına hizala
            bar.get_height() + 0.01,           # Çubuğun biraz üstüne yaz
            f'{bar.get_height():.3f}',          # 3 ondalık basamakla göster
            ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance — Random Forest hangi özelliğe ne kadar önem verdi?
importances = rf_model.feature_importances_
# feature_importances_ → Random Forest'ın kendi hesapladığı, her özelliğin karar vermede ne kadar etkili olduğu
# Değerler 0-1 arasında, toplamı 1 eder

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(features, importances, color='salmon')  # barh() → yatay çubuk grafik (h = horizontal)
ax.set_title('Random Forest — Özellik Önemi')
ax.set_xlabel('Önem Skoru')

for i, v in enumerate(importances):  # enumerate() → hem indeksi (i) hem değeri (v) ver
    ax.text(v + 0.002, i, f'{v:.3f}', va='center')  # Her çubuğun yanına sayısal değerini yaz

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Ek — Keşifsel Veri Analizi

In [ ]:
top_cats = df['Category'].value_counts().head(10)
# value_counts() → her kategoride kaç uygulama var, çoktan aza sırala
# .head(10) → ilk 10'unu al

fig, ax = plt.subplots(figsize=(10, 5))
top_cats.plot(kind='bar', ax=ax, color='teal', edgecolor='white')
ax.set_title('En Çok Uygulama Bulunan 10 Kategori')
ax.set_xlabel('Kategori')
ax.set_ylabel('Uygulama Sayısı')
plt.xticks(rotation=45, ha='right')  # Etiketleri 45 derece eğik yaz, sağa hizala (çakışmasın)
plt.tight_layout()
plt.savefig('category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. GÜN — Sonuç

| Model | Accuracy | F1-Score (Weighted) |
|-------|----------|---------------------|
| Logistic Regression | 0.7876 | 0.6965 |
| **Random Forest** | **0.7663** | **0.7544** |

**Sonuç:** Random Forest, F1-Score metriğinde belirgin şekilde öne çıktı (0.7544 > 0.6965). Veri setinde başarılı uygulama sayısı başarısızdan çok daha fazla olduğu için Accuracy yanıltıcı olabilir — F1-Score bu dengesizliği hesaba katar ve daha güvenilir bir ölçüt sunar.